"""
DecodeLabs – Data Science Project 1
Advanced EDA & Feature Engineering Pipeline
IPO Architecture: Input → Process → Output
"""

# Libraries and reading dataset

In [1]:
import pandas as pd
import numpy as np
import pandera as pa
from pandera import Column, DataFrameSchema, Check
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

In [6]:
df = pd.read_csv("Dataset for Data Analytics - Sheet1.csv", parse_dates=["Date"])

In [7]:
print("ORIGINAL DATASET")
print(f"  Shape        : {df.shape}")
print(f"\nMissing values (%):")
miss_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(miss_pct[miss_pct > 0].to_string())

ORIGINAL DATASET
  Shape        : (1200, 14)

Missing values (%):
CouponCode    25.75


# MODULE 1 – INPUT: Securing Fidelity

In [2]:
print("MODULE 1 – MISSING DATA & OUTLIER HANDLING")
 
# Missing Data Decision Matrix 
# CouponCode: 25.75% missing → >20% threshold → KNN Imputation
 
print("\n[Imputation] CouponCode (25.75% missing) → KNN (>20% rule)")
 
le = LabelEncoder()
df["CouponCode_encoded"] = le.fit_transform(df["CouponCode"].fillna("MISSING"))
 
imputer = KNNImputer(n_neighbors=5)
df["CouponCode_encoded"] = imputer.fit_transform(df[["CouponCode_encoded"]])
 
df["CouponCode_encoded"] = (
    df["CouponCode_encoded"]
    .round()
    .astype(int)
    .clip(0, len(le.classes_) - 1)
)
df["CouponCode"] = le.inverse_transform(df["CouponCode_encoded"])
df.drop(columns=["CouponCode_encoded"], inplace=True)
 
print(f"  Remaining nulls: {df['CouponCode'].isnull().sum()}")
print(f"  CouponCode values: {df['CouponCode'].unique()}")

MODULE 1 – MISSING DATA & OUTLIER HANDLING

[Imputation] CouponCode (25.75% missing) → KNN (>20% rule)


NameError: name 'df' is not defined

In [ ]:
# Outlier Detection & Winsorization (IQR)

numeric_cols = ["UnitPrice", "TotalPrice", "Quantity", "ItemsInCart"]
 
print("\n[Outlier] IQR Winsorization (numpy.clip) results:")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers_before = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(f"  {col:15s} | lower={lower:.2f}  upper={upper:.2f} | "f"outliers capped: {outliers_before}")

# MODULE 2 – PROCESS: Vectorized Computation Engine

In [ ]:
print("MODULE 2 – FEATURE ENGINEERING (vectorized, no loops)")
 
# ── Feature 1: Revenue Per Item ───────────────
df["revenue_per_item"] = df["TotalPrice"] / df["Quantity"]
print("\n[Feature 1] revenue_per_item = TotalPrice / Quantity")
 
# ── Feature 2: Price-to-Cart Ratio ───────────
df["price_to_cart_ratio"] = df["UnitPrice"] / df["ItemsInCart"]
print("[Feature 2] price_to_cart_ratio = UnitPrice / ItemsInCart")
 
# ── Feature 3: Order Month ───────────────────
df["order_month"] = df["Date"].dt.month.astype("int64")
print("[Feature 3] order_month = month extracted from Date")
 
# ── Feature 4: Has Coupon (binary flag) ──────
df["has_coupon"] = (~df["CouponCode"].isin(["MISSING", ""])).astype(int)
print("[Feature 4] has_coupon = 1 if valid coupon applied, else 0")
 
# ── Feature 5: Discount Tier ─────────────────
discount_map = {"SAVE10": 10.0, "WINTER15": 15.0, "FREESHIP": 0.0, "MISSING": 0.0}
df["discount_tier"] = df["CouponCode"].map(discount_map).fillna(0.0)
print("[Feature 5] discount_tier = estimated % discount from coupon")